# Training an LSTM-based Authorized Push Payment (APP) Fraud Detection Model
This notebook is used to prepare and preprocess the datasets from IBM Synthetic Datasets, which are then used for training an LSTM-based Authorized Push Payment (APP) fraud detection model. The datasets are loaded from four different CSV files and then combined into one DataFrame. This enriched dataset provides the features needed to train a fraud detection model that can learn patterns from account characteristics, bank metrics, and transaction behaviors. Such as:
- Transaction details (amount, currency, format, type)
- Fraud labels (Is_APP_Fraud)
- Sender account features (country, currency, entity type, overdraft, branch/bank metrics)
- Recipient account features (country, entity type)
The notebook then builds a preprocessing + LSTM model, trains it, and exports a self-contained ONNX file for inference.


In [ ]:
import os, io, sys
import pandas as pd
import numpy as np
import tensorflow as tf
import keras
from keras import layers, models, ops

print(f"TensorFlow {tf.__version__}  |  Keras {keras.__version__}")

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ["WRAPT_DISABLE_EXTENSIONS"] = "true"

datasets_dir   = "./datasets/"
model_save_dir = './saved_model'
os.makedirs(model_save_dir, exist_ok=True)

timesteps       = 7     # transactions per sequence
batch_size      = 2048
test_batch_size = 2000



## Data Preparation


This code loads and enriches bank transfer transaction data for fraud detection by joining three datasets: banks, accounts (people + companies), and transactions. It creates a comprehensive dataset where each transaction is enriched with contextual features from both the sender's account (account type, currency, branch size, bank statistics) and recipient's account (country, entity type), while filtering out cash transactions and converting data types for efficient processing. The final merged dataset contains transaction details plus account-level and bank-level features that help the LSTM model identify fraudulent Authorized Push Payment (APP) patterns.

In [ ]:
print("Loading datasets...")

banks_cols     = ["Bank_ID", "Num_Transactions", "Num_Total_Locations"]
accts_cols     = ["Financial_Institution_ID", "Financial_Institution_Name", "Branch",
                  "Account_ID", "Account_Country", "Account_Currency", "Entity_Type",
                  "Account_Type", "Max_Overdraft"]
bank_xfers_cols = ["Transaction_Number", "Transaction_Date", "Transaction_Time",
                   "Transaction_Day_of_Week", "From_Bank", "From_Account", "To_Bank",
                   "To_Account", "Amount_Paid", "Payment_Currency", "Payment_Format",
                   "Transaction_Type", "Is_Instant_Payments_Fraud", "Is_APP_Fraud",
                   "Sufficient_Funds", "Overdraft_Okay", "Is_Hold", "From_End_Balance"]

banks_df           = pd.read_csv(datasets_dir + "us_small_banks.csv",               usecols=banks_cols)
accts_people_df    = pd.read_csv(datasets_dir + "us_small_liquid_accts_people.csv",  usecols=accts_cols)
accts_companies_df = pd.read_csv(datasets_dir + "us_small_liquid_accts_companies.csv",
                                 encoding='iso8859_2', usecols=accts_cols)
bank_xfers_df      = pd.read_csv(datasets_dir + "us_small_bank_xfers-chrono.csv",   usecols=bank_xfers_cols)

accts_df = pd.concat([accts_people_df, accts_companies_df], ignore_index=True)
accts_df = accts_df.astype({"Account_Country": "category", "Account_Currency": "category",
                             "Entity_Type": "category", "Account_Type": "category", "Account_ID": str})

branches_df = (accts_df.groupby(["Financial_Institution_ID", "Branch"])["Account_ID"]
               .nunique().to_frame(name="Branch_Account_Count").reset_index())
branches_df = branches_df.merge(banks_df[["Bank_ID", "Num_Transactions", "Num_Total_Locations"]],
                                left_on="Financial_Institution_ID", right_on="Bank_ID", how="left")
branches_df.drop("Bank_ID", axis="columns", inplace=True)
branches_df.rename(columns={"Num_Transactions": "Bank_Num_Transactions",
                             "Num_Total_Locations": "Bank_Num_Total_Locations"}, inplace=True)
branches_df = branches_df.astype({"Bank_Num_Transactions": "Int64", "Bank_Num_Total_Locations": "Int64"})

accts_merged_df = accts_df.merge(branches_df, on=["Financial_Institution_ID", "Branch"], how="left")

bank_xfers_cleaned_df = bank_xfers_df[
    (bank_xfers_df["From_Bank"] != "Cash") & (bank_xfers_df["To_Bank"] != "Cash")
].astype({"From_Account": str, "To_Account": str, "Payment_Currency": "category",
           "Payment_Format": "category", "Transaction_Type": "category",
           "Is_APP_Fraud": bool, "Sufficient_Funds": bool, "Overdraft_Okay": bool, "Is_Hold": bool})
bank_xfers_cleaned_df.reset_index(drop=True, inplace=True)

from_cols = ["Financial_Institution_Name", "Account_ID", "Account_Country", "Account_Currency",
             "Entity_Type", "Account_Type", "Max_Overdraft",
             "Branch_Account_Count", "Bank_Num_Transactions", "Bank_Num_Total_Locations"]
bank_xfers_merged_df = bank_xfers_cleaned_df.merge(
    accts_merged_df[from_cols],
    left_on=["From_Bank", "From_Account"],
    right_on=["Financial_Institution_Name", "Account_ID"], how="left")
bank_xfers_merged_df.drop(["Financial_Institution_Name", "Account_ID"], axis="columns", inplace=True)
bank_xfers_merged_df.rename(columns={
    "Account_Country": "From_Account_Country", "Account_Currency": "From_Account_Currency",
    "Entity_Type": "From_Entity_Type", "Account_Type": "From_Account_Type",
    "Max_Overdraft": "From_Account_Max_Overdraft",
    "Branch_Account_Count": "From_Branch_Account_Count",
    "Bank_Num_Transactions": "From_Bank_Num_Transactions",
    "Bank_Num_Total_Locations": "From_Bank_Num_Total_Locations"}, inplace=True)

bank_xfers_merged_df["Timestamp"] = (bank_xfers_merged_df["Transaction_Date"].astype(str) + " " +
                                     bank_xfers_merged_df["Transaction_Time"].astype(str))

bank_xfers_merged_df = bank_xfers_merged_df.merge(
    accts_merged_df[["Financial_Institution_Name", "Account_ID", "Account_Country", "Entity_Type"]],
    left_on=["To_Bank", "To_Account"],
    right_on=["Financial_Institution_Name", "Account_ID"], how="left")
bank_xfers_merged_df.drop(["Financial_Institution_Name", "Account_ID"], axis="columns", inplace=True)
bank_xfers_merged_df.rename(columns={"Account_Country": "To_Account_Country",
                                     "Entity_Type": "To_Entity_Type"}, inplace=True)
bank_xfers_merged_df.reset_index(drop=True, inplace=True)

dataset_df = bank_xfers_merged_df
print(f"Dataset shape: {dataset_df.shape}  |  Fraud cases: {dataset_df['Is_APP_Fraud'].sum()}")



## Feature Engineering


In [ ]:
print("Applying transformations...")

dataset_df["Timestamp"]   = pd.to_datetime(dataset_df["Timestamp"], format="%Y-%m-%d %H:%M:%S.%f")
dataset_df["Month"]       = dataset_df["Timestamp"].dt.month
dataset_df["Day_Of_Month"]= dataset_df["Timestamp"].dt.day
dataset_df["Day_Of_Week"] = dataset_df["Timestamp"].dt.dayofweek
dataset_df["Hour"]        = dataset_df["Timestamp"].dt.hour
dataset_df["Minute"]      = dataset_df["Timestamp"].dt.minute
dataset_df["Is_Weekday"]  = dataset_df["Timestamp"].dt.dayofweek.isin([0,1,2,3,4]).astype(int)

for col in ["From_Account_Max_Overdraft", "From_Branch_Account_Count",
            "From_Bank_Num_Transactions", "From_Bank_Num_Total_Locations"]:
    dataset_df[col] = dataset_df[col].fillna(0)

categorical_cols = ["From_Entity_Type", "From_Account_Type", "To_Entity_Type",
                    "Payment_Format", "Transaction_Type", "From_Account_Country",
                    "From_Account_Currency", "To_Account_Country", "Payment_Currency"]
for col in categorical_cols:
    if isinstance(dataset_df[col].dtype, pd.CategoricalDtype):
        if "Unknown" not in dataset_df[col].cat.categories:
            dataset_df[col] = dataset_df[col].cat.add_categories(["Unknown"])
    dataset_df[col] = dataset_df[col].fillna("Unknown").astype(str)

print("Feature engineering complete!")



### Custom Keras Layers for Preprocessing
The custom preprocessing layers (CyclicalEncoding, LogTransform, TimeOfDayEncoding) are needed to enable the model to understand temporal patterns in fraud detection (e.g., fraud more common at certain times/days) while keeping everything in a single deployable model.
ONNX-compatible replacements for `StringLookup` (which uses TF hash tables that tf2onnx cannot export). All layers use only `tf.equal` / `tf.cast` — opset-13 primitives.


In [ ]:
@keras.saving.register_keras_serializable()
class OnnxVocabOneHot(layers.Layer):
    """String → float32 one-hot (replaces StringLookup output_mode='one_hot')."""

    def __init__(self, vocab_size=None, name=None, **kwargs):
        super().__init__(name=name, **kwargs)
        self._vocab = None
        self._vocab_size = vocab_size

    def compute_output_spec(self, inputs):
        return keras.KerasTensor(shape=(inputs.shape[0], self._vocab_size), dtype='float32')

    def adapt(self, data):
        vals = sorted(tf.unique(tf.reshape(data, [-1]))[0].numpy().tolist(), key=lambda b: b.decode())
        if b'Unknown' in vals:
            vals.remove(b'Unknown')
            vals = [b'Unknown'] + vals
        self._vocab = tf.constant(vals, dtype=tf.string)
        self._vocab_size = len(vals)

    def set_vocabulary(self, vocab):
        vocab = [v.encode() if isinstance(v, str) else v for v in vocab]
        self._vocab = tf.constant(vocab, dtype=tf.string)
        self._vocab_size = len(vocab)

    def call(self, inputs):
        return tf.cast(tf.squeeze(tf.equal(inputs[:, :, tf.newaxis], self._vocab), axis=1), tf.float32)

    def get_config(self):
        cfg = super().get_config()
        if self._vocab is not None:
            cfg['vocabulary'] = [v.decode() for v in self._vocab.numpy().tolist()]
        return cfg

    @classmethod
    def from_config(cls, config):
        vocab = config.pop('vocabulary', None)
        obj = cls(**config)
        if vocab is not None:
            obj.set_vocabulary(vocab)
        return obj


@keras.saving.register_keras_serializable()
class OnnxVocabOrdinal(layers.Layer):
    """String → float32 ordinal index (replaces StringLookup output_mode='int')."""

    def __init__(self, name=None, **kwargs):
        super().__init__(name=name, **kwargs)
        self._vocab = None

    def compute_output_spec(self, inputs):
        return keras.KerasTensor(shape=(inputs.shape[0], 1), dtype='float32')

    def adapt(self, data):
        vals = sorted(tf.unique(tf.reshape(data, [-1]))[0].numpy().tolist(), key=lambda b: b.decode())
        if b'Unknown' in vals:
            vals.remove(b'Unknown')
            vals = [b'Unknown'] + vals
        self._vocab = tf.constant(vals, dtype=tf.string)

    def set_vocabulary(self, vocab):
        vocab = [v.encode() if isinstance(v, str) else v for v in vocab]
        self._vocab = tf.constant(vocab, dtype=tf.string)

    def call(self, inputs):
        matches = tf.squeeze(tf.equal(inputs[:, :, tf.newaxis], self._vocab), axis=1)
        return tf.cast(tf.expand_dims(tf.argmax(tf.cast(matches, tf.int32), axis=-1), -1), tf.float32)

    def get_config(self):
        cfg = super().get_config()
        if self._vocab is not None:
            cfg['vocabulary'] = [v.decode() for v in self._vocab.numpy().tolist()]
        return cfg

    @classmethod
    def from_config(cls, config):
        vocab = config.pop('vocabulary', None)
        obj = cls(**config)
        if vocab is not None:
            obj.set_vocabulary(vocab)
        return obj


@keras.saving.register_keras_serializable()
class CyclicalEncoding(layers.Layer):
    """Encodes a periodic feature as (sin, cos) pair."""
    def __init__(self, max_value, name=None, **kwargs):
        super().__init__(name=name, **kwargs)
        self.max_value = float(max_value)
    def call(self, inputs):
        angle = 2 * np.pi * inputs / self.max_value
        return ops.concatenate([ops.sin(angle), ops.cos(angle)], axis=-1)
    def get_config(self):
        return {**super().get_config(), "max_value": self.max_value}


@keras.saving.register_keras_serializable()
class LogTransform(layers.Layer):
    """Sign-preserving log1p transform."""
    def call(self, inputs):
        return ops.sign(inputs) * ops.log(ops.abs(inputs) + 1.0)
    def get_config(self):
        return super().get_config()


@keras.saving.register_keras_serializable()
class TimeOfDayEncoding(layers.Layer):
    """Encodes (hour, minute) as a (sin, cos) time-of-day pair."""
    def call(self, inputs):
        seconds = inputs[:, 0:1] * 3600 + inputs[:, 1:2] * 60
        angle   = 2 * np.pi * seconds / 86400
        return ops.concatenate([ops.sin(angle), ops.cos(angle)], axis=-1)
    def get_config(self):
        return super().get_config()



## Build a Keras model that includes all preprocessing layers


In [ ]:
def build_preprocessing_model():
    """Functional Keras model: 24 raw inputs → normalised feature vector."""
    f32 = tf.float32
    fstr = tf.string
    inp = lambda name, dt=f32: layers.Input(shape=(1,), dtype=dt, name=name)

    inputs = {
        'Month':                     inp('Month'),
        'Day_Of_Month':              inp('Day_Of_Month'),
        'Day_Of_Week':               inp('Day_Of_Week'),
        'Hour':                      inp('Hour'),
        'Minute':                    inp('Minute'),
        'Is_Weekday':                inp('Is_Weekday'),
        'Sufficient_Funds':          inp('Sufficient_Funds'),
        'Overdraft_Okay':            inp('Overdraft_Okay'),
        'Is_Hold':                   inp('Is_Hold'),
        'Amount_Paid':               inp('Amount_Paid'),
        'From_End_Balance':          inp('From_End_Balance'),
        'From_Account_Max_Overdraft':inp('From_Account_Max_Overdraft'),
        'From_Branch_Account_Count': inp('From_Branch_Account_Count'),
        'From_Bank_Num_Transactions':inp('From_Bank_Num_Transactions'),
        'From_Bank_Num_Total_Locations': inp('From_Bank_Num_Total_Locations'),
        'From_Entity_Type':          inp('From_Entity_Type',  fstr),
        'From_Account_Type':         inp('From_Account_Type', fstr),
        'To_Entity_Type':            inp('To_Entity_Type',    fstr),
        'Payment_Format':            inp('Payment_Format',    fstr),
        'Transaction_Type':          inp('Transaction_Type',  fstr),
        'From_Account_Country':      inp('From_Account_Country',   fstr),
        'From_Account_Currency':     inp('From_Account_Currency',  fstr),
        'To_Account_Country':        inp('To_Account_Country',     fstr),
        'Payment_Currency':          inp('Payment_Currency',       fstr),
    }
    processed = []

    # Cyclical temporal features
    cyclical = layers.Concatenate(name='cyclical_concat')([
        CyclicalEncoding(12, name='month_cyclical')(inputs['Month']),
        CyclicalEncoding(31, name='day_cyclical')(inputs['Day_Of_Month']),
        CyclicalEncoding(7,  name='dow_cyclical')(inputs['Day_Of_Week']),
        TimeOfDayEncoding(name='time_of_day_cyclical')(
            layers.Concatenate(name='time_concat')([inputs['Hour'], inputs['Minute']])),
    ])
    processed.append(layers.Normalization(name='cyclical_norm')(cyclical))

    # Boolean pass-through
    processed.append(layers.Concatenate(name='boolean_concat')([
        inputs['Is_Weekday'], inputs['Sufficient_Funds'],
        inputs['Overdraft_Okay'], inputs['Is_Hold'],
    ]))

    # Log-transformed amounts
    amounts = layers.Concatenate(name='amounts_concat')([
        LogTransform(name='amount_paid_log')(inputs['Amount_Paid']),
        LogTransform(name='balance_log')(inputs['From_End_Balance']),
        LogTransform(name='overdraft_log')(inputs['From_Account_Max_Overdraft']),
    ])
    processed.append(layers.Normalization(name='amounts_norm')(amounts))

    # Count features
    counts = layers.Concatenate(name='counts_concat')([
        inputs['From_Branch_Account_Count'],
        inputs['From_Bank_Num_Transactions'],
        inputs['From_Bank_Num_Total_Locations'],
    ])
    processed.append(layers.Normalization(name='counts_norm')(counts))

    # Low-cardinality → one-hot
    for feat in ['From_Entity_Type', 'From_Account_Type', 'To_Entity_Type',
                 'Payment_Format', 'Transaction_Type']:
        processed.append(OnnxVocabOneHot(name=f'{feat}_lookup')(inputs[feat]))

    # High-cardinality → ordinal + normalise
    for feat in ['From_Account_Country', 'From_Account_Currency',
                 'To_Account_Country', 'Payment_Currency']:
        processed.append(
            layers.Normalization(name=f'{feat}_norm')(
                OnnxVocabOrdinal(name=f'{feat}_lookup')(inputs[feat])))

    out = layers.Concatenate(name='feature_concat')(processed)
    return models.Model(inputs=inputs, outputs=out, name='preprocessing'), inputs


preprocessing_model, feature_inputs = build_preprocessing_model()
print("Preprocessing model built.")
preprocessing_model.summary()



### Adapt preprocessing layers with actual data
Adapting preprocessing layers with actual data is essential because these layers need to learn statistics from your training data to work correctly.

In [ ]:
print("Adapting preprocessing layers...")

def prepare_data_dict(df):
    """DataFrame → dict of TF tensors keyed by feature name."""
    f32 = tf.float32
    d = {
        'Month':                      tf.constant(df['Month'].values.reshape(-1,1),                         dtype=f32),
        'Day_Of_Month':               tf.constant(df['Day_Of_Month'].values.reshape(-1,1),                  dtype=f32),
        'Day_Of_Week':                tf.constant(df['Day_Of_Week'].values.reshape(-1,1),                   dtype=f32),
        'Hour':                       tf.constant(df['Hour'].values.reshape(-1,1),                          dtype=f32),
        'Minute':                     tf.constant(df['Minute'].values.reshape(-1,1),                        dtype=f32),
        'Is_Weekday':                 tf.constant(df['Is_Weekday'].values.reshape(-1,1),                    dtype=f32),
        'Sufficient_Funds':           tf.constant(df['Sufficient_Funds'].astype(int).values.reshape(-1,1),  dtype=f32),
        'Overdraft_Okay':             tf.constant(df['Overdraft_Okay'].astype(int).values.reshape(-1,1),    dtype=f32),
        'Is_Hold':                    tf.constant(df['Is_Hold'].astype(int).values.reshape(-1,1),           dtype=f32),
        'Amount_Paid':                tf.constant(df['Amount_Paid'].values.reshape(-1,1),                   dtype=f32),
        'From_End_Balance':           tf.constant(df['From_End_Balance'].values.reshape(-1,1),              dtype=f32),
        'From_Account_Max_Overdraft': tf.constant(df['From_Account_Max_Overdraft'].values.reshape(-1,1),    dtype=f32),
        'From_Branch_Account_Count':  tf.constant(df['From_Branch_Account_Count'].values.reshape(-1,1),     dtype=f32),
        'From_Bank_Num_Transactions': tf.constant(df['From_Bank_Num_Transactions'].values.reshape(-1,1),    dtype=f32),
        'From_Bank_Num_Total_Locations': tf.constant(df['From_Bank_Num_Total_Locations'].values.reshape(-1,1), dtype=f32),
    }
    for col in categorical_cols:
        d[col] = tf.constant(df[col].values.reshape(-1,1), dtype=tf.string)
    return d

sample_data = prepare_data_dict(dataset_df.iloc[:min(10000, len(dataset_df))])

vocab_map = {feat: sample_data[feat] for feat in categorical_cols}
for layer in preprocessing_model.layers:
    if isinstance(layer, (OnnxVocabOneHot, OnnxVocabOrdinal)):
        for feat, data in vocab_map.items():
            if feat in layer.name:
                layer.adapt(data)
                print(f"  Adapted {layer.name}")
                break

cyclical_sample = layers.Concatenate()([
    CyclicalEncoding(12)(sample_data['Month']),
    CyclicalEncoding(31)(sample_data['Day_Of_Month']),
    CyclicalEncoding(7)(sample_data['Day_Of_Week']),
    TimeOfDayEncoding()(layers.Concatenate()([sample_data['Hour'], sample_data['Minute']]))
])
preprocessing_model.get_layer('cyclical_norm').adapt(cyclical_sample)

amounts_sample = layers.Concatenate()([
    LogTransform()(sample_data['Amount_Paid']),
    LogTransform()(sample_data['From_End_Balance']),
    LogTransform()(sample_data['From_Account_Max_Overdraft'])
])
preprocessing_model.get_layer('amounts_norm').adapt(amounts_sample)

counts_sample = layers.Concatenate()([
    sample_data['From_Branch_Account_Count'],
    sample_data['From_Bank_Num_Transactions'],
    sample_data['From_Bank_Num_Total_Locations']
])
preprocessing_model.get_layer('counts_norm').adapt(counts_sample)

print("  Adapted cyclical_norm, amounts_norm, counts_norm")

test_output = preprocessing_model(sample_data)
preprocessed_feature_size = test_output.shape[-1]
print(f"Normalization adapted. Preprocessed feature size: {preprocessed_feature_size}")



## Build LSTM Model
This code builds a complete fraud detection model by creating sequence inputs for 7 timesteps of transaction history, individually applying the preprocessing model to each timestep (since TimeDistributed doesn't support dictionary inputs), and concatenating the results into a properly shaped tensor for the LSTM layers. The architecture consists of two stacked LSTM layers (200 units each) that process the preprocessed sequences to capture temporal patterns, followed by a sigmoid output layer that predicts fraud probability for each timestep. The entire model (preprocessing + LSTM) is unified into a single Keras model that can be exported to ONNX for deployment.

In [ ]:
# Sequence inputs: (batch, timesteps, feature_dim)
sequence_inputs = {
    key: layers.Input(shape=(timesteps,) + feature_inputs[key].shape[1:],
                      dtype=feature_inputs[key].dtype, name=key)
    for key in feature_inputs
}

# Apply preprocessing per timestep, then concatenate
preprocessed_timesteps = []
for t in range(timesteps):
    t_inputs = {k: ops.squeeze(sequence_inputs[k][:, t:t+1, :], axis=1) for k in sequence_inputs}
    preprocessed_timesteps.append(ops.expand_dims(preprocessing_model(t_inputs), axis=1))

preprocessed_sequences = layers.Reshape(
    (timesteps, preprocessed_feature_size), name='reshape_sequences')(
    layers.Concatenate(axis=1, name='concat_timesteps')(preprocessed_timesteps))

lstm_out = layers.LSTM(200, return_sequences=True, name='lstm_1')(preprocessed_sequences)
lstm_out = layers.LSTM(200, return_sequences=True, name='lstm_2')(lstm_out)
output   = layers.Dense(1, activation='sigmoid', name='output')(lstm_out)

complete_model = models.Model(inputs=sequence_inputs, outputs=output, name='fraud_detection_lstm')
print("Complete model built.")
complete_model.summary()



## Prepare Training Data


In [ ]:
print("Preparing training data...")

labels       = dataset_df['Is_APP_Fraud'].values.astype(np.float32)
total        = len(labels)
train_size   = int(total * 0.5)
val_size     = int(total * 0.3)
train_indices = np.arange(train_size)
val_indices   = np.arange(train_size, train_size + val_size)
test_indices  = np.arange(train_size + val_size, total)

print(f"Split — Train: {len(train_indices)}  Val: {len(val_indices)}  Test: {len(test_indices)}")

train_label_slice = labels[train_indices]
genuine_count = int((train_label_slice == 0).sum())
fraud_count = int((train_label_slice == 1).sum())
positive_class_weight = float(genuine_count / fraud_count)
print(f"Training class distribution — genuine: {genuine_count}  fraud: {fraud_count}")
print(f"Using positive class weight={positive_class_weight}")

def create_sequences_dict(df, indices, labels, timesteps):
    """Build windowed sequence arrays from a DataFrame."""
    valid = indices[indices >= timesteps - 1]
    if len(valid) == 0:
        return None, None
    full = prepare_data_dict(df)
    seq_data = {}
    for key, tensor in full.items():
        feat = tensor.numpy()
        seqs = (np.empty  if feat.dtype == np.object_ else np.zeros)(
            (len(valid), timesteps) + feat.shape[1:],
            dtype=feat.dtype if feat.dtype != np.object_ else object)
        for i, idx in enumerate(valid):
            for t in range(timesteps):
                seqs[i, t] = feat[idx - (timesteps - 1 - t)]
        seq_data[key] = (tf.constant(seqs, dtype=tf.string)
                         if seqs.dtype == object else seqs)
    seq_labels = np.zeros((len(valid), timesteps, 1), dtype=np.float32)
    for i, idx in enumerate(valid):
        for t in range(timesteps):
            seq_labels[i, t, 0] = labels[idx - (timesteps - 1 - t)]
    return seq_data, seq_labels


print("Creating sequences...")
train_seq_data, train_seq_labels = create_sequences_dict(dataset_df, train_indices, labels, timesteps)
val_seq_data,   val_seq_labels   = create_sequences_dict(dataset_df, val_indices,   labels, timesteps)
test_seq_data,  test_seq_labels  = create_sequences_dict(dataset_df, test_indices,  labels, timesteps)
print(f"Train {train_seq_labels.shape}  Val {val_seq_labels.shape}  Test {test_seq_labels.shape}")



## Compile and Train


In [ ]:
# Custom metrics — evaluate only the last (most recent) timestep
class TP(keras.metrics.TruePositives):
    def update_state(self, y_true, y_pred, sample_weight=None):
        super().update_state(y_true[:,-1,:], y_pred[:,-1,:], sample_weight)

class FP(keras.metrics.FalsePositives):
    def update_state(self, y_true, y_pred, sample_weight=None):
        super().update_state(y_true[:,-1,:], y_pred[:,-1,:], sample_weight)

class FN(keras.metrics.FalseNegatives):
    def update_state(self, y_true, y_pred, sample_weight=None):
        super().update_state(y_true[:,-1,:], y_pred[:,-1,:], sample_weight)

class TN(keras.metrics.TrueNegatives):
    def update_state(self, y_true, y_pred, sample_weight=None):
        super().update_state(y_true[:,-1,:], y_pred[:,-1,:], sample_weight)

complete_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', TP(name='TP'), FP(name='FP'), FN(name='FN'), TN(name='TN'),
             keras.metrics.TruePositives(name='tp'), keras.metrics.FalsePositives(name='fp'),
             keras.metrics.FalseNegatives(name='fn'), keras.metrics.TrueNegatives(name='tn')]
)

checkpoint_dir = "./checkpoints/app_fraud_lstm_keras_preprocessing/"
os.makedirs(checkpoint_dir, exist_ok=True)
cp_callback = keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_dir + "iter-{epoch:02d}.weights.h5",
    save_weights_only=True, verbose=1)

print("Training model...")
train_sample_weights = np.where(train_seq_labels == 1.0, positive_class_weight, 1.0).astype(np.float32)
val_sample_weights = np.where(val_seq_labels == 1.0, positive_class_weight, 1.0).astype(np.float32)

history = complete_model.fit(
    train_seq_data, train_seq_labels,
    batch_size=batch_size, epochs=1,
    validation_data=(val_seq_data, val_seq_labels, val_sample_weights),
    callbacks=[cp_callback], verbose=1,
    sample_weight=train_sample_weights)



## Save and Evaluate


In [ ]:
print(f"Saving model to {model_save_dir}...")
complete_model.save(os.path.join(model_save_dir, "fraud_detection_model.keras"))
complete_model.save_weights(os.path.join(model_save_dir, "model_weights.weights.h5"))

print("Evaluating on test set...")
test_results = complete_model.evaluate(test_seq_data, test_seq_labels,
                                        batch_size=test_batch_size, verbose=1)
for name, value in zip(complete_model.metrics_names, test_results):
    print(f"  {name}: {value:.4f}")





## Export to ONNX
`tf2onnx.convert.from_keras` was removed in TF 2.16 / Keras 3. We wrap the model in a `tf.Module` so `from_function` can trace through a single dict input.


In [ ]:
print("Exporting unified model to ONNX...")

import tf2onnx
import onnx
from onnx import numpy_helper, TensorProto

onnx_output_path = os.path.join(model_save_dir, "fraud_detection_unified.onnx")

serving_name_map = {
    'Month': 'month',
    'Day_Of_Month': 'day_of_month',
    'Day_Of_Week': 'day_of_week',
    'Hour': 'hour',
    'Minute': 'minute',
    'Is_Weekday': 'is_weekday',
    'Sufficient_Funds': 'sufficient_funds',
    'Overdraft_Okay': 'overdraft_okay',
    'Is_Hold': 'is_hold',
    'Amount_Paid': 'amount_paid',
    'From_End_Balance': 'from_end_balance',
    'From_Account_Max_Overdraft': 'from_account_max_overdraft',
    'From_Branch_Account_Count': 'from_branch_account_count',
    'From_Bank_Num_Transactions': 'from_bank_num_transactions',
    'From_Bank_Num_Total_Locations': 'from_bank_num_total_locations',
    'From_Entity_Type': 'from_entity_type',
    'From_Account_Type': 'from_account_type',
    'To_Entity_Type': 'to_entity_type',
    'Payment_Format': 'payment_format',
    'Transaction_Type': 'transaction_type',
    'From_Account_Country': 'from_account_country',
    'From_Account_Currency': 'from_account_currency',
    'To_Account_Country': 'to_account_country',
    'Payment_Currency': 'payment_currency',
}

input_spec = {serving_name_map[inp.name]: tf.TensorSpec(shape=[None] + list(inp.shape[1:]),
                                                        dtype=inp.dtype, name=serving_name_map[inp.name])
              for inp in complete_model.inputs}

class _ServingWrapper(tf.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    @tf.function(input_signature=[input_spec])
    def __call__(self, inputs):
        keras_inputs = {inp.name: inputs[serving_name_map[inp.name]] for inp in self.model.inputs}
        return self.model(keras_inputs, training=False)

serving_wrapper = _ServingWrapper(complete_model)
model_proto, _ = tf2onnx.convert.from_function(
    serving_wrapper.__call__,
    input_signature=[input_spec], opset=13, output_path=onnx_output_path)

# ── Post-process: fix two tf2onnx export bugs ──────────────────────────────────
#
# Bug 1 — Leaked preprocessing constants
#   OnnxVocabOneHot._vocab and Normalization mean/variance are traced as free
#   graph inputs instead of embedded constants, so every serving call fails
#   with "Required inputs … are missing from input feed".
#
# Bug 2 — Invalid Cast(STRING→INT32) in vocab lookup subgraphs
#   tf2onnx converts tf.equal(string, vocab) as Cast(str→INT32)→Equal(int,int).
#   ONNX Cast STRING→INT32 tries to parse strings as decimal integers and
#   crashes at runtime ("stoll: no conversion").
#
# Fix: replace every broken Slice→Cast→Equal→Squeeze→Cast subgraph with
#   ai.onnx.ml LabelEncoder + OneHotEncoder (one-hot) or Reshape+Cast (ordinal),
#   then embed normalization mean/variance as ONNX initializers.
# ──────────────────────────────────────────────────────────────────────────────

print("Post-processing ONNX graph: fixing tf2onnx export bugs...")

# 1. Collect vocab and norm stats from the Keras preprocessing layers
_layer_vocab, _layer_is_onehot, _layer_norm = {}, {}, {}
for layer in preprocessing_model.layers:
    if isinstance(layer, OnnxVocabOneHot) and layer._vocab is not None:
        _layer_vocab[layer.name]     = [v.decode() for v in layer._vocab.numpy()]
        _layer_is_onehot[layer.name] = True
    elif isinstance(layer, OnnxVocabOrdinal) and layer._vocab is not None:
        _layer_vocab[layer.name]     = [v.decode() for v in layer._vocab.numpy()]
        _layer_is_onehot[layer.name] = False
    elif isinstance(layer, layers.Normalization):
        weights = {w.name.split('/')[-1]: w.numpy() for w in layer.weights}
        mk = next((k for k in weights if 'mean'     in k), None)
        vk = next((k for k in weights if 'variance' in k), None)
        if mk and vk:
            _layer_norm[layer.name] = (weights[mk].reshape(1,-1).astype(np.float32),
                                       weights[vk].reshape(1,-1).astype(np.float32))

# 2. Reload proto and ensure ai.onnx.ml opset is declared
model_proto = onnx.load(onnx_output_path)
graph = model_proto.graph
if not any(op.domain == 'ai.onnx.ml' for op in model_proto.opset_import):
    oi = model_proto.opset_import.add(); oi.domain = 'ai.onnx.ml'; oi.version = 3

# 3. Replace broken vocab lookup subgraphs with LabelEncoder + OHE/Cast
out_to_node = {o: n for n in graph.node for o in n.output}
ONNX_PREFIX        = "fraud_detection_lstm_1/preprocessing_1"
leaked_vocab_names = {inp.name for inp in graph.input if 'Equal/y:0' in inp.name}

def _keras_name(leaked):
    lws = leaked.split(f"{ONNX_PREFIX}/")[-1].split('/')[0]
    return lws[:-2] if lws.endswith('_1') else lws

_new_nodes, _to_remove = [], set()
_replaced = 0

for leaked_name, keras_name in sorted({n: _keras_name(n) for n in leaked_vocab_names}.items()):
    if keras_name not in _layer_vocab:
        print(f"  WARNING: no vocab for {keras_name!r}"); continue
    vocab_tokens = _layer_vocab[keras_name]
    is_onehot    = _layer_is_onehot[keras_name]
    vocab_size   = len(vocab_tokens)

    vc_node = next((n for n in graph.node if n.op_type=='Cast' and n.input[0]==leaked_name), None)
    if not vc_node: continue
    vc_out  = vc_node.output[0]

    for eq_node in [n for n in graph.node if n.op_type=='Equal' and vc_out in n.input]:
        fc_out  = eq_node.input[0] if eq_node.input[1]==vc_out else eq_node.input[1]
        fc_node = out_to_node.get(fc_out)
        if not fc_node or fc_node.op_type != 'Cast': continue
        sl_node = out_to_node.get(fc_node.input[0])
        if not sl_node or sl_node.op_type != 'Slice': continue
        un_node = out_to_node.get(sl_node.input[0])
        if not un_node or un_node.op_type != 'Unsqueeze': continue
        str_in = un_node.input[0]   # [batch,1] string tensor — our new input

        sq_node = next((n for n in graph.node if n.op_type=='Squeeze' and eq_node.output[0] in n.input), None)
        if not sq_node: continue
        ps_node = next((n for n in graph.node if sq_node.output[0] in n.input), None)
        if not ps_node or ps_node.op_type != 'Cast': continue

        ax_node = next((n for n in graph.node if n.op_type=='ArgMax' and ps_node.output[0] in n.input), None)
        marks   = [fc_node, sl_node, un_node, eq_node, sq_node, ps_node]

        if ax_node:
            ex_node = next((n for n in graph.node if n.op_type=='Unsqueeze' and ax_node.output[0] in n.input), None)
            c1_node = next((n for n in graph.node if n.op_type=='Cast'      and ex_node.output[0] in n.input), None) if ex_node else None
            if not ex_node or not c1_node: continue
            final_out = c1_node.output[0]
            marks.extend([ax_node, ex_node, c1_node])
        else:
            final_out = ps_node.output[0]

        sid       = f"_vf_{_replaced}"
        sh_name   = f"{sid}_sh";  rs_out = f"{sid}_rs"
        graph.initializer.append(numpy_helper.from_array(np.array([-1], dtype=np.int64), name=sh_name))
        _new_nodes.append(onnx.helper.make_node('Reshape', [str_in, sh_name], [rs_out], name=f"{sid}_R"))
        le_out = f"{sid}_le"
        _new_nodes.append(onnx.helper.make_node('LabelEncoder', [rs_out], [le_out],
            name=f"{sid}_LE", domain='ai.onnx.ml',
            keys_strings=vocab_tokens, values_int64s=list(range(vocab_size)), default_int64=-1))

        if ax_node is None:
            _new_nodes.append(onnx.helper.make_node('OneHotEncoder', [le_out], [final_out],
                name=f"{sid}_OHE", domain='ai.onnx.ml',
                cats_int64s=list(range(vocab_size)), zeros=1))
        else:
            sh2 = f"{sid}_sh2"; r2 = f"{sid}_r2"
            graph.initializer.append(numpy_helper.from_array(np.array([-1,1], dtype=np.int64), name=sh2))
            _new_nodes.append(onnx.helper.make_node('Reshape', [le_out, sh2], [r2],       name=f"{sid}_R2"))
            _new_nodes.append(onnx.helper.make_node('Cast',    [r2], [final_out],          name=f"{sid}_CF", to=TensorProto.FLOAT))

        for n in marks: _to_remove.add(n.name)
        _replaced += 1

for n in graph.node:
    if n.op_type == 'Cast' and n.input[0] in leaked_vocab_names:
        _to_remove.add(n.name)

# Rebuild node list in topological order
all_nodes = [n for n in graph.node if n.name not in _to_remove] + _new_nodes
produced  = ({i.name for i in graph.input} | {i.name for i in graph.initializer})
sorted_nodes, remaining = [], list(all_nodes)
for _ in range(len(remaining) + 1):
    ready = [n for n in remaining if all(i=='' or i in produced for i in n.input)]
    if not ready: ready = [remaining[0]]
    for n in ready:
        sorted_nodes.append(n); produced.update(n.output)
    remaining = [n for n in remaining if n not in ready]
    if not remaining: break
sorted_nodes.extend(remaining)
del graph.node[:]; graph.node.extend(sorted_nodes)

# Rebuild graph.input without leaked vocab and normalization constants, keeping
# the serving input names already present on the in-memory graph.
feature_inputs_only = [inp for inp in list(graph.input)
                       if 'Equal/y:0' not in inp.name and 'Sub/y:0' not in inp.name and 'Sqrt/x:0' not in inp.name]
del graph.input[:]
graph.input.extend(feature_inputs_only)

print(f"  Replaced {_replaced} vocab subgraphs ({len(_to_remove)} nodes removed, {len(_new_nodes)} added).")

# 4. Embed normalization mean/variance as ONNX initializers (Bug 1)
graph_value_info = list(graph.input) + list(graph.value_info) + list(graph.output)
graph_tensor_names = {vi.name for vi in graph_value_info} | {n for node in graph.node for n in node.input}
_norm_feed = {}
for ln, (m, v) in _layer_norm.items():
    for tensor_name in sorted(n for n in graph_tensor_names
                              if n.endswith(f"/{ln}_1/Sub/y:0") or n.endswith(f"/{ln}_1/Sqrt/x:0")):
        if tensor_name.endswith('/Sub/y:0'):
            _norm_feed[tensor_name] = m
        else:
            _norm_feed[tensor_name] = v

for name in sorted(_norm_feed):
    if not any(init.name == name for init in graph.initializer):
        graph.initializer.append(numpy_helper.from_array(_norm_feed[name], name=name))
new_inp = [i for i in graph.input if i.name not in _norm_feed]
del graph.input[:]
graph.input.extend(new_inp)

# Re-sort after embedding normalization constants because those graph inputs are
# replaced with initializers after the earlier node-order rebuild.
produced = ({i.name for i in graph.input} | {i.name for i in graph.initializer})
sorted_nodes, remaining = [], list(graph.node)
for _ in range(len(remaining) + 1):
    ready = [n for n in remaining if all(i == '' or i in produced for i in n.input)]
    if not ready:
        ready = [remaining[0]]
    for n in ready:
        sorted_nodes.append(n)
        produced.update(n.output)
    remaining = [n for n in remaining if n not in ready]
    if not remaining:
        break
sorted_nodes.extend(remaining)
del graph.node[:]
graph.node.extend(sorted_nodes)

print(f"  Embedded {len(_norm_feed)} norm constants. Remaining graph inputs: {len(graph.input)}")

onnx.checker.check_model(model_proto)
onnx.save(model_proto, onnx_output_path)

print(f"✓ Unified model exported to {onnx_output_path}")
print("  Includes: preprocessing, LSTM layers, output layer")